# Week 5 - Project 2 Completion: Sentiment Analysis with LSTM
Intern: Nishkarsh Jandial | Portal ID: ICP-E2ABBE74-2026

**Dataset:** IMDB Movie Reviews (50,000 reviews — positive/negative)
**This week:** LSTM model using TensorFlow/Keras, final comparison with baseline models (Logistic Regression + Naive Bayes from Week 4), evaluation, and documentation.

## 1. Install and Import Libraries

In [ ]:
!pip install datasets tensorflow -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import string
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

print('TensorFlow version:', tf.__version__)
print('All libraries imported successfully.')

## 2. Load and Preprocess Data

In [ ]:
from datasets import load_dataset

dataset = load_dataset('imdb')
train_df = pd.DataFrame(dataset['train'])
test_df = pd.DataFrame(dataset['test'])

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r'<.*?>', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()
    return text

train_df['clean_text'] = train_df['text'].apply(preprocess_text)
test_df['clean_text'] = test_df['text'].apply(preprocess_text)

print('Data loaded and preprocessed.')
print('Train shape:', train_df.shape)
print('Test shape:', test_df.shape)

## 3. Reproduce Baseline Models from Week 4

In [ ]:
tfidf = TfidfVectorizer(max_features=10000, stop_words='english', ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(train_df['clean_text'])
X_test_tfidf = tfidf.transform(test_df['clean_text'])

y_train = train_df['label']
y_test = test_df['label']

# Logistic Regression
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_tfidf, y_train)
y_pred_lr = lr.predict(X_test_tfidf)
acc_lr = accuracy_score(y_test, y_pred_lr)

# Naive Bayes
nb = MultinomialNB()
nb.fit(X_train_tfidf, y_train)
y_pred_nb = nb.predict(X_test_tfidf)
acc_nb = accuracy_score(y_test, y_pred_nb)

print(f'Logistic Regression Accuracy: {acc_lr:.4f}')
print(f'Naive Bayes Accuracy: {acc_nb:.4f}')

## 4. Prepare Data for LSTM

In [ ]:
MAX_WORDS = 10000
MAX_LEN = 200

# Tokenize
tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(train_df['clean_text'])

# Convert to sequences
X_train_seq = tokenizer.texts_to_sequences(train_df['clean_text'])
X_test_seq = tokenizer.texts_to_sequences(test_df['clean_text'])

# Pad sequences to same length
X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, truncating='post', padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, truncating='post', padding='post')

y_train_arr = np.array(y_train)
y_test_arr = np.array(y_test)

print('Train padded shape:', X_train_pad.shape)
print('Test padded shape:', X_test_pad.shape)

## 5. Build LSTM Model

In [ ]:
model = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=64, input_length=MAX_LEN),
    Bidirectional(LSTM(64, return_sequences=False)),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

## 6. Train LSTM Model

In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = model.fit(
    X_train_pad, y_train_arr,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

## 7. Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('LSTM Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('LSTM Loss')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()

## 8. Evaluate LSTM on Test Set

In [ ]:
loss, acc_lstm = model.evaluate(X_test_pad, y_test_arr, verbose=0)
print(f'LSTM Test Accuracy: {acc_lstm:.4f}')

y_pred_lstm = (model.predict(X_test_pad) > 0.5).astype(int).flatten()
print(classification_report(y_test_arr, y_pred_lstm, target_names=['Negative', 'Positive']))

## 9. Confusion Matrix - LSTM

In [ ]:
cm = confusion_matrix(y_test_arr, y_pred_lstm)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Negative', 'Positive'],
            yticklabels=['Negative', 'Positive'])
plt.title('Confusion Matrix - LSTM')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

## 10. Final Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Logistic Regression (TF-IDF)', 'Naive Bayes (TF-IDF)', 'Bidirectional LSTM'],
    'Accuracy': [round(acc_lr, 4), round(acc_nb, 4), round(acc_lstm, 4)]
})

print(results.to_string(index=False))

plt.figure(figsize=(7, 4))
sns.barplot(x='Model', y='Accuracy', data=results, palette='Blues_d')
plt.title('Model Accuracy Comparison - Sentiment Analysis')
plt.ylim(0.8, 1.0)
plt.xticks(rotation=10)
plt.show()

## 11. Test on Custom Reviews

In [ ]:
def predict_sentiment(review):
    cleaned = preprocess_text(review)
    seq = tokenizer.texts_to_sequences([cleaned])
    padded = pad_sequences(seq, maxlen=MAX_LEN, truncating='post', padding='post')
    prob = model.predict(padded, verbose=0)[0][0]
    label = 'POSITIVE' if prob > 0.5 else 'NEGATIVE'
    return f'{label} (confidence: {prob:.2f})'

reviews = [
    "This movie was absolutely fantastic! The acting was superb.",
    "Terrible film. Complete waste of time, very boring and disappointing.",
    "It was okay, not great but not bad either."
]

for r in reviews:
    print(f'Review: {r[:60]}...')
    print(f'Prediction: {predict_sentiment(r)}\n')

## 12. Summary and Conclusions

**Project 2: Sentiment Analysis on IMDB Reviews — COMPLETED**

- **Best Model:** Bidirectional LSTM — captures sequential context in text
- **Key Findings:**
  - TF-IDF + Logistic Regression is a strong baseline, fast and interpretable
  - LSTM captures word order and context, improving accuracy over bag-of-words approaches
  - Bidirectional LSTM reads text in both directions for richer understanding
  - EarlyStopping prevented overfitting during training
- **Next:** Week 6 — Final portfolio documentation, GitHub cleanup, and submission